Here is how it works:
- Shallow Clone from all branches and Config files: .yml, yaml and related .json, .sh
- The url list is sorted and locked for random sampling or stop/run to continue incrementally 
- START_NUMBER holds the last reviewed url's index
- The sample repos are stored in "Cloned_Sample"
- This will save the sample repos as well as metrics, configs, builds and test lines
- metadata includes the key metrics and project name


In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import glob
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests

# === CONFIGURATION ===
MAX_PROJECTS = 5
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER"))
print("Start_number:", START_NUMBER)
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
base_dir = Path(r"E:\Android Mobile Project\AndroidProjects")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_csv = base_dir / "8.2-Project_Metadata.csv"

# === ENSURE FOLDERS EXIST ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir, cloned_sample_dir]:
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df = df.sort_values(by='github_url').reset_index(drop=True)
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

metadata_rows = []

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.loc[i, 'github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    # --- Shallow clone all branches ---
    try:
        subprocess.run([
            'git', 'clone', '--depth', '1', '--no-single-branch', url, str(repo_path)
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("✅ Shallow clone of all branches complete")
    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout while cloning {repo_name}, skipping...")
        continue

    # --- Extract relevant config and test-related files (.yml, .yaml, .sh, .json) ---
    extracted_count = 0
    for root, _, files in os.walk(repo_path):
        for file in files:
            lower_file = file.lower()
            if lower_file.endswith(('.yml', '.yaml', '.sh', '.json')):
                file_path = Path(root) / file
                rel_path = file_path.relative_to(repo_path)
                rel_path_str = str(rel_path).lower()

                # .sh/.json must include keywords like test, instrument, or ci
                if lower_file.endswith(('.sh', '.json')) and not any(x in rel_path_str for x in ['test', 'instrument', 'ci']):
                    continue

                safe_name = f"{repo_name}.{str(rel_path).replace(os.sep, '_')}"
                shutil.copy2(file_path, yml_output_dir / safe_name)
                extracted_count += 1

    print(f"📄 Extracted {extracted_count} relevant files (.yml/.yaml/.sh/.json)")

    # --- Collect basic metadata ---
    metadata_rows.append({
        'project_name': repo_name
    })

    # --- Save updated START_NUMBER ---
    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

# === SAVE METADATA CSV ===
if metadata_rows:
    pd.DataFrame(metadata_rows).to_csv(metadata_csv, index=False)
    print(f"\n✅ Saved metadata for {len(metadata_rows)} projects to {metadata_csv}")

print("\n🏁 Finished processing selected projects.")


Start_number: 1793
🔁 Loaded SAMPLE_LIST from .env with 150 indices.
Start_number:  1793

🔍 [1793/1796] Processing 1792.zugaldia.android-robocar...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: 1792.zugaldia.android-robocar

🔍 [1794/1796] Processing 1793.zulip.zulip-mobile...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: 1793.zulip.zulip-mobile

🔍 [1795/1796] Processing 1794.zulkarnine.WordleSolver...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
❌ Error handling repo folder for 1794.zulkarnine.WordleSolver: Destination path 'E:\Android Mobile Project\AndroidProjects\Cloned_Sample\1794.zulkarnine.WordleSolver\1794.zulkarnine.WordleSolver' already